# Data Access Layer — Manual Testing

This notebook tests the data access layer against simulated case data.

**Prerequisites:**
- `pip install -r requirements.txt`
- Generate data: `python -m data --output data/simulated/ --seed 42 --cases 5`

## 1. Setup — Load Gateway & Catalog

In [16]:
import os, sys

# Resolve project root (parent of notebooks/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'notebooks':
    # Already at project root (e.g. VS Code sets cwd to workspace root)
    PROJECT_ROOT = os.getcwd()

# Add project root to Python path
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'simulated')
PROFILE_DIR = os.path.join(PROJECT_ROOT, 'config', 'data_profiles')

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir: {DATA_DIR}')
print(f'Profile dir: {PROFILE_DIR}')
print(f'Data dir exists: {os.path.isdir(DATA_DIR)}')
print(f'Profile dir exists: {os.path.isdir(PROFILE_DIR)}')

Project root: /Users/mingxuanliu/Library/CloudStorage/GoogleDrive-mingxuan99michelle@gmail.com/My Drive/Projs/AgenticSys_v1
Data dir: /Users/mingxuanliu/Library/CloudStorage/GoogleDrive-mingxuan99michelle@gmail.com/My Drive/Projs/AgenticSys_v1/data/simulated
Profile dir: /Users/mingxuanliu/Library/CloudStorage/GoogleDrive-mingxuan99michelle@gmail.com/My Drive/Projs/AgenticSys_v1/config/data_profiles
Data dir exists: True
Profile dir exists: True


In [17]:
from data.gateway import SimulatedDataGateway
from data.catalog import DataCatalog
from tools.data_tools import init_tools, list_available_tables, get_table_schema, query_table

# Load gateway from case folders, catalog from YAML profiles
gw = SimulatedDataGateway.from_case_folders(DATA_DIR)
catalog = DataCatalog(profile_dir=PROFILE_DIR)
init_tools(gw, catalog)

print(f'Gateway loaded: {len(gw.list_case_ids())} cases')
print(f'Catalog loaded: {len(catalog.list_tables())} tables')

Gateway loaded: 5 cases
Catalog loaded: 11 tables


## 2. List Available Cases

In [18]:
cases = gw.list_case_ids()
print(f'Available cases ({len(cases)}): {cases}')

Available cases (5): ['CASE-00001', 'CASE-00002', 'CASE-00003', 'CASE-00004', 'CASE-00005']


## 3. Select a Case & List Tables

In [19]:
CASE_ID = 'CASE-00001'  # Change this to test other cases
gw.set_case(CASE_ID)
print(list_available_tables())

Tables for case CASE-00001:
bureau_full
bureau_trades
cross_bu
cust_tenure
income_dti
model_scores
pmts_detail
score_drivers
txn_monthly
wcc_flags
xbu_summary


## 4. Inspect Table Schema

In [20]:
import json

print('=== bureau_full schema ===')
print(json.dumps(json.loads(get_table_schema('bureau_full')), indent=2))

=== bureau_full schema ===
{
  "case_id": {
    "type": "string",
    "description": "Unique case identifier"
  },
  "fico_score": {
    "type": "int",
    "description": "FICO Score \u2014 industry-standard credit risk score predicting likelihood of serious delinquency using traditional credit bureau data (payment history, balances, utilization). Values below 721 are risky."
  },
  "sbfe_score": {
    "type": "int",
    "description": "SBFE Score \u2014 business credit score predicting likelihood of financial delinquency or failure. Values below 863 are risky."
  },
  "ln_credit_score": {
    "type": "int",
    "description": "LexisNexis Credit Score \u2014 risk score predicting consumer likelihood of default using traditional credit bureau data and public records. Values below 681 are risky."
  },
  "ln_blended_score": {
    "type": "int",
    "description": "LexisNexis Blended Score \u2014 combines traditional credit data with alternative data (public records, non-credit signals) to

In [21]:
print('=== cross_bu schema ===')
print(json.dumps(json.loads(get_table_schema('cross_bu')), indent=2))

=== cross_bu schema ===
{
  "case_id": {
    "type": "string",
    "description": "Foreign key to case"
  },
  "card_name": {
    "type": "categorical",
    "description": "Name of the card used by the customer"
  },
  "card_type": {
    "type": "categorical",
    "description": "Type of card \u2014 Charge (pay-in-full), Lending (revolving credit), or LOC (line of credit)"
  },
  "card_portfolio": {
    "type": "categorical",
    "description": "Amex portfolio to which the card belongs"
  },
  "card_limit": {
    "type": "float",
    "description": "Credit card limit in USD. Charge cards may have no preset spending limit (NPSL) \u2014 represented as high values."
  },
  "balance": {
    "type": "float",
    "description": "Outstanding exposure on the card in USD"
  },
  "card_tenure": {
    "type": "int",
    "description": "Tenure of the card account in months"
  },
  "account_status": {
    "type": "categorical",
    "description": "Account status \u2014 'Current' means up to date, 3

## 5. Query Bureau Data

In [ ]:
print(f'=== Bureau Full (first 3 rows) — {CASE_ID} ===')
print(query_table('bureau_full', limit=3))

## 6. Query Cross-BU Cards

In [ ]:
print(f'=== All Cards — {CASE_ID} ===')
print(query_table('cross_bu'))

## 7. Filter — Find Delinquent Cards

In [22]:
print('=== Delinquent Cards (90dpb) ===')
print(query_table('cross_bu', filter_column='account_status', filter_value='90dpb'))

print()
print('=== Delinquent Cards (30dpb) ===')
print(query_table('cross_bu', filter_column='account_status', filter_value='30dpb'))

=== Delinquent Cards (90dpb) ===
[
  {
    "card_name": "Business Green Card",
    "card_type": "charge",
    "card_portfolio": "Corporate",
    "card_limit": "26745.7325",
    "balance": "0.0",
    "card_tenure": "34",
    "account_status": "90dpb",
    "past_delinquencies_12m": "0",
    "merchant_name": "BlueSky Travel Agency",
    "merchant_industry": "Real Estate",
    "merchant_charge_volume": "58539.5681"
  }
]

=== Delinquent Cards (30dpb) ===
No rows matching filter in 'cross_bu'.


## 8. Query Spend Transactions

In [23]:
print(f'=== Spend Transactions (first 5) — {CASE_ID} ===')
print(query_table('pmts_detail', limit=5))

=== Spend Transactions (first 5) — CASE-00001 ===
[
  {
    "date": "2023-03-07",
    "month": "August",
    "amount": "2019.1053",
    "merchant_name": "Sysco Foods",
    "merchant_industry": "Airlines",
    "merchant_risk_score": "0.2326",
    "spend_concentration": "1.5202",
    "rnn_spend_score": "0.0038",
    "spend_divergence_index": "1.9065",
    "customer_industry": "Professional Services"
  },
  {
    "date": "2024-09-19",
    "month": "September",
    "amount": "9805.8943",
    "merchant_name": "Shell Oil",
    "merchant_industry": "Airlines",
    "merchant_risk_score": "0.548",
    "spend_concentration": "1.7652",
    "rnn_spend_score": "0.0103",
    "spend_divergence_index": "1.4404",
    "customer_industry": "Food Services & Restaurants"
  },
  {
    "date": "2024-01-20",
    "month": "February",
    "amount": "4739.0686",
    "merchant_name": "Delta Airlines",
    "merchant_industry": "",
    "merchant_risk_score": "0.3717",
    "spend_concentration": "1.2919",
    "rnn_s

## 9. Query Model Scores

In [ ]:
print(f'=== Model Scores (first row, first 20 fields) — {CASE_ID} ===')
result = query_table('model_scores', limit=1)
data = json.loads(result)
if data:
    row = data[0]
    for i, (k, v) in enumerate(row.items()):
        if i >= 20:
            print(f'  ... and {len(row) - 20} more fields')
            break
        print(f'  {k}: {v}')
else:
    print('No model scores found')

## 10. Query Score Drivers

In [24]:
print(f'=== Score Drivers (first 5) — {CASE_ID} ===')
print(query_table('score_drivers', limit=5))

=== Score Drivers (first 5) — CASE-00001 ===
[
  {
    "model_id": "collections_v1",
    "driver_rank": "5",
    "driver_variable": "last_cycle_cut_revolve_rate",
    "driver_direction": "increasing_risk",
    "driver_value": "-0.91",
    "driver_contribution": "-0.053",
    "driver_description": "Low bureau score"
  },
  {
    "model_id": "collections_v1",
    "driver_rank": "4",
    "driver_variable": "avutil_exrvlv_balgt50",
    "driver_direction": "increasing_risk",
    "driver_value": "0.32",
    "driver_contribution": "0.0027",
    "driver_description": "High credit loss probability"
  },
  {
    "model_id": "risk_v3",
    "driver_rank": "2",
    "driver_variable": "tot_struct_risk_score",
    "driver_direction": "increasing_risk",
    "driver_value": "-0.8219",
    "driver_contribution": "0.0781",
    "driver_description": "High credit loss probability"
  },
  {
    "model_id": "risk_v3",
    "driver_rank": "3",
    "driver_variable": "tot_struct_risk_score",
    "driver_directi

## 11. Query Other Tables

In [ ]:
for table in ['cust_tenure', 'income_dti', 'wcc_flags', 'xbu_summary', 'txn_monthly']:
    print(f'=== {table} ===')
    print(query_table(table, limit=3))
    print()

## 12. Edge Cases

In [25]:
# Missing table
print('=== Missing Table ===')
print(query_table('nonexistent_table'))

print()
# Filter with no matches
print('=== Filter with No Matches ===')
print(query_table('cross_bu', filter_column='account_status', filter_value='999dpb'))

print()
# Missing schema
print('=== Missing Schema ===')
print(get_table_schema('nonexistent_table'))

=== Missing Table ===
Data unavailable: table 'nonexistent_table' not found for current case.

=== Filter with No Matches ===
No rows matching filter in 'cross_bu'.

=== Missing Schema ===
Data unavailable


## 13. Switch Case & Compare

In [26]:
# Compare bureau data across cases
for case_id in gw.list_case_ids():
    gw.set_case(case_id)
    result = query_table('bureau_full', limit=1)
    data = json.loads(result)
    if data:
        row = data[0]
        print(f'{case_id}:  FICO={row.get("fico_score", "N/A")}  '
              f'SBFE={row.get("sbfe_score", "N/A")}  '
              f'Delinq Trades={row.get("delinquent_external_trades", "N/A")}  '
              f'Exposure=${row.get("overall_external_exposure", "N/A")}  '
              f'Judgements={row.get("judgements_org_count", "N/A")}  '
              f'Liens={row.get("lien_org_count", "N/A")}')

# Reset to original case
gw.set_case(CASE_ID)

CASE-00001:  FICO=737  SBFE=884  Delinq Trades=0  Exposure=$27298.5715  Judgements=1  Liens=3
CASE-00002:  FICO=793  SBFE=894  Delinq Trades=0  Exposure=$103201.8231  Judgements=0  Liens=0
CASE-00003:  FICO=739  SBFE=831  Delinq Trades=2  Exposure=$50259.4327  Judgements=1  Liens=1
CASE-00004:  FICO=785  SBFE=865  Delinq Trades=0  Exposure=$55329.1529  Judgements=0  Liens=0
CASE-00005:  FICO=700  SBFE=861  Delinq Trades=1  Exposure=$107588.7209  Judgements=0  Liens=0


## 14. Full Data Catalog

In [27]:
print(catalog.to_prompt_context())

=== DATA CATALOG ===

TABLE: bureau_full
  Full bureau credit file — one row per case. Aggregated bureau snapshot including FICO, business credit scores, external exposure, and legal filings.
  - fico_score [int] (range: 300–850): FICO Score — industry-standard credit risk score predicting likelihood of serious delinquency using traditional credit bureau data (payment history, balances, utilization). Values below 721 are risky.
  - sbfe_score [int] (range: 500–950): SBFE Score — business credit score predicting likelihood of financial delinquency or failure. Values below 863 are risky.
  - ln_credit_score [int] (range: 300–900): LexisNexis Credit Score — risk score predicting consumer likelihood of default using traditional credit bureau data and public records. Values below 681 are risky.
  - ln_blended_score [int] (range: 300–900): LexisNexis Blended Score — combines traditional credit data with alternative data (public records, non-credit signals) to assess credit risk more holistic